In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import gc
import yaml
import logging
from collections import defaultdict, deque
from typing import Any, List, Tuple, Optional
import re
import math

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True 
)


In [ ]:
class ConnectionManager():
    def __init__(self, db_con_str):
        # la fonction duckdb.connect transforme le path en absolue, autant le faire ici 
        p = Path(db_con_str).expanduser().resolve()
        p.parent.mkdir(parents=True, exist_ok=True) # s'assurer que toute l'arboressence est créée

        self.db_con_str = p
        
        # connexion lazy
        self._con = None


    @property
    def con(self):
        if(self._con is None):
            # se connecter à la base de données
            self._con = duckdb.connect(self.db_con_str)

        return self._con


    @con.setter
    def con(self, value):
        # si au moment de changer la connexion on a déjà une connexion active
        if(self._con is not None):
            self._con.close() # cloturer la connexion en cours
            self._con = None # retirer la référence sur la connexion en cours
            gc.collect() # appeler le garbage collector pour forcer l'action de libérer les ressources et éviter les conflits d'accès

        self._con = value # pointer sur la nouvelle connexion
    
    
    def close_con(self):
        # on exploite le setter de la propriété pour cloturer correctement la connexion
        self.con = None


    def __del__(self):
        """Ferme automatiquement la connexion DuckDB quand l'objet est détruit."""
        try:
            self.close_con()
        except Exception:
            pass


    def __enter__(self):
        return self
    

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close_con()
        return False

In [ ]:
class ConnectionUtils(ConnectionManager):
    def __init__(self, db_con_str : str):
        super().__init__(db_con_str)


    def tables(self):
        """Retourne la liste de toutes les tables physiques de la base courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            AND table_type = 'BASE TABLE'
            ORDER BY table_name
        """)


    def views(self):
        """Cette fonction renvoi la liste de toutes vues accessibles dans la base de donnees courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.views
            WHERE table_catalog = current_database()
            ORDER BY table_name
        """)


    def tables_views(self):
        """Retourne la liste des tables et des vues de la base courante"""
        return self.con.sql("""
            SELECT 
                table_name,
                table_type
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            ORDER BY table_type, table_name
        """)


    def table_exists(self, table_name : str):
        """Cette fonction verifie qu'une table physique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                table_name = '{table_name}'
                AND table_type = 'BASE TABLE'
                AND table_catalog = current_database()
        """).fetchone()[0]


    def view_exists(self, view_name : str):
        """Cette fonction verifie qu'une vue existe"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.views 
            WHERE 
                table_name = '{view_name}'
                AND table_catalog = current_database()
        """).fetchone()[0]


    def table_view_exists(self, name : str):
        """Cette fonction verifie qu'une table physique ou une vue logique existe"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                table_name = '{name}'
                AND (table_type = 'BASE TABLE' OR table_type = 'VIEW')
                AND table_catalog = current_database()
        """).fetchone()[0]


    def drop_table_if_exists(self, table_name : str):
        """Cette fonction permet de supprimer une table si elle existe"""
        self.con.sql(f"DROP TABLE IF EXISTS {table_name}")


    def drop_tables_if_exists(self, tables : list[str]):
        """Cette fonction supprime chaque table de la liste si elle existe"""
        for table_name in tables:
            self.drop_table_if_exists(table_name)


    def drop_view_if_exists(self, view_name : str):
        """Cette fonction permet de supprimer une vue si elle existe"""
        self.con.sql(f"DROP VIEW IF EXISTS {view_name}")


    def drop_views_if_exists(self, views : list[str]):
        """Cette fonction supprime chaque vue de la liste si elle existe"""
        for view_name in views:
            self.drop_view_if_exists(view_name)


    def table(self, table_name : str):
        """Cette fonction renvoi la table dont le nom est passe en parametre"""
        return self.con.table(table_name)


    def view(self, view_name : str):
        """Cette fonction renvoi la vue dont le nom est passe en parametre"""
        return self.con.view(view_name)


    def table_view(self, name : str):
        """Retourne la relation d'une table ou d'une vue selon ce qui existe"""
        return self.con.sql(f"SELECT * FROM {name}")


    def create_table_view(
        self,
        name : str,
        sql : str,
        type : str = "view",
        mode : str = "create_if_not_exists"
    ):
        """Cree ou remplace une table ou une vue selon le mode demande"""
        sql = sql.strip().rstrip(";")
        object_type = type.upper()

        if object_type not in {"TABLE", "VIEW"}:
            raise ValueError(f"Type non supporte : {type}")

        if mode == "create_if_not_exists":
            self.con.sql(
                f"CREATE {object_type} IF NOT EXISTS {name} AS ({sql})"
            )

        elif mode == "create_or_replace":
            self.con.sql(
                f"CREATE OR REPLACE {object_type} {name} AS ({sql})"
            )

        else:
            raise ValueError(f"Mode non supporte : {mode}")


    def execute_sql(self, sql : str):
        """Execute une requete SQL sans creer de table ou de vue"""
        sql = sql.strip().rstrip(";")
        return self.con.sql(sql)


In [ ]:
class DependencyTree:
    def __init__(self, data: dict[str, dict[str, Any]]):
        """
        data : dictionnaire de la forme
        {
            "v_sales": {"requires": ["t_sales"], ...},
            "t_sales": {"requires": ["df_sales"], ...},
            ...
        }
        """
        self.data = data
        self.graph = self._build_graph()          # node -> list of dependencies
        self.reverse_graph = self._build_reverse_graph()  # node -> list of dependents

    def _build_graph(self) -> dict[str, list[str]]:
        return {
            name: config.get("requires", [])
            for name, config in self.data.items()
        }

    def _build_reverse_graph(self) -> dict[str, list[str]]:
        reverse = defaultdict(list)
        for node, deps in self.graph.items():
            for dep in deps:
                reverse[dep].append(node)
        return dict(reverse)

    # -------------------------------------------------------------------------
    # Informations de base
    # -------------------------------------------------------------------------
    def nodes(self) -> list[str]:
        """Retourne tous les nœuds du graphe."""
        return list(self.graph.keys())

    def dependencies(self, name: str) -> list[str]:
        """Retourne les dépendances directes d'un nœud."""
        return self.graph.get(name, [])

    def dependents(self, name: str) -> list[str]:
        """Retourne les nœuds qui dépendent directement de celui-ci."""
        return self.reverse_graph.get(name, [])

    def roots(self) -> list[str]:
        """Nœuds qui ne sont requis par personne."""
        all_deps = {dep for deps in self.graph.values() for dep in deps}
        return [n for n in self.graph if n not in all_deps]

    def leaves(self) -> list[str]:
        """Nœuds qui n'ont aucune dépendance."""
        return [n for n, deps in self.graph.items() if not deps]

    # -------------------------------------------------------------------------
    # Dépendances récursives
    # -------------------------------------------------------------------------
    def all_dependencies(self, name: str) -> list[str]:
        """Retourne toutes les dépendances (directes + indirectes) dans l'ordre topologique."""
        result = []
        visited = set()

        def dfs(node: str):
            if node in visited:
                return
            visited.add(node)
            for dep in self.graph.get(node, []):
                dfs(dep)
            result.append(node)

        dfs(name)
        return result[:-1]  # on retire le nœud lui-même

    def creation_order(self, name: str | None = None) -> list[str]:
        """
        Ordre de création (topologique).
        Si name est fourni → uniquement pour ce nœud et ses dépendances.
        Sinon → ordre global.
        """
        if name:
            nodes = self.all_dependencies(name) + [name]
        else:
            nodes = self.nodes()

        in_degree = {n: 0 for n in nodes}
        for n in nodes:
            for dep in self.graph.get(n, []):
                if dep in in_degree:
                    in_degree[n] += 1

        queue = deque([n for n, deg in in_degree.items() if deg == 0])
        order = []

        while queue:
            node = queue.popleft()
            order.append(node)
            for dependent in self.reverse_graph.get(node, []):
                if dependent in in_degree:
                    in_degree[dependent] -= 1
                    if in_degree[dependent] == 0:
                        queue.append(dependent)

        return order

    # -------------------------------------------------------------------------
    # Affichage
    # -------------------------------------------------------------------------
    def print_tree(self, root: str | None = None):
        """Affiche l'arbre de dépendances en texte."""
        def _print(node: str, prefix: str = "", is_last: bool = True, visited: set | None = None):
            if visited is None:
                visited = set()

            connector = "└── " if is_last else "├── "
            print(f"{prefix}{connector}{node}")

            if node in visited:
                print(f"{prefix}{'    ' if is_last else '│   '}└── [cycle détecté]")
                return

            visited = visited | {node}
            deps = self.graph.get(node, [])
            new_prefix = prefix + ("    " if is_last else "│   ")

            for i, dep in enumerate(deps):
                _print(dep, new_prefix, i == len(deps) - 1, visited)

        if root:
            print(f"\n=== Dépendances de '{root}' ===\n")
            _print(root)
        else:
            print("\n=== Graphe complet ===\n")
            roots = self.roots()
            for i, r in enumerate(roots):
                _print(r, is_last=(i == len(roots) - 1))

    def print_levels(self, name: str | None = None):
        """Affiche les nœuds par niveau topologique."""
        order = self.creation_order(name)
        print(f"\n=== Ordre de création {'de ' + name if name else 'global'} ===\n")
        for i, node in enumerate(order, 1):
            print(f"{i:2d}. {node}")

In [ ]:
class ConnectionPipeline(ConnectionUtils):
    def __init__(self, db_con_str : str, pipeline_file_path : str):
        super().__init__(db_con_str)

        p = Path(pipeline_file_path).expanduser().resolve()
        p.parent.mkdir(parents=True, exist_ok=True)

        self.pipeline_file_path = p
        self._pipeline = None
        self._tree = None


    def load_pipeline(self) -> dict:
        """
        Charge la definition du pipeline.

        Si le chemin est un fichier YAML, charge ce fichier.
        Si le chemin est un dossier, charge tous les fichiers YAML recursivement.
        """
        path = self.pipeline_file_path

        if not path.exists():
            raise FileNotFoundError(
                f"Le chemin du pipeline n'existe pas : {path}"
            )

        if path.is_file():
            if path.suffix.lower() not in {".yaml", ".yml"}:
                raise ValueError(
                    f"Le fichier pipeline doit etre un fichier YAML : {path}"
                )

            with path.open("r", encoding="utf-8") as file:
                pipeline = yaml.safe_load(file) or {}

            if not isinstance(pipeline, dict):
                raise ValueError(
                    f"Le contenu YAML doit etre un dictionnaire : {path}"
                )

            return pipeline

        yaml_files = sorted(
            [
                *path.rglob("*.yaml"),
                *path.rglob("*.yml"),
            ],
            key=lambda file_path: str(file_path.relative_to(path)),
        )

        if not yaml_files:
            raise FileNotFoundError(
                f"Aucun fichier YAML trouve dans : {path}"
            )

        merged_pipeline : dict = {}
        object_sources : dict[str, Path] = {}

        for yaml_file in yaml_files:
            with yaml_file.open("r", encoding="utf-8") as file:
                current_pipeline = yaml.safe_load(file) or {}

            if not isinstance(current_pipeline, dict):
                raise ValueError(
                    f"Le contenu YAML doit etre un dictionnaire : {yaml_file}"
                )

            for object_name, config in current_pipeline.items():
                if object_name in merged_pipeline:
                    previous_file = object_sources[object_name]

                    raise ValueError(
                        f"L'objet '{object_name}' est defini plusieurs fois : "
                        f"'{previous_file}' et '{yaml_file}'."
                    )

                merged_pipeline[object_name] = config
                object_sources[object_name] = yaml_file

        return merged_pipeline


    @property
    def pipeline(self):
        if self._pipeline is None:
            self._pipeline = self.load_pipeline()
        return self._pipeline


    @property
    def tree(self):
        if self._tree is None:
            self._tree = DependencyTree(self.pipeline)
        return self._tree


    def df_from_file(self, file : str | Path, **kwargs) -> pd.DataFrame:
        """Charge un fichier en DataFrame selon son extension"""
        path = Path(file).expanduser().resolve()
        suffix = path.suffix.lower()

        if suffix in {".xlsx", ".xls", ".xlsm"}:
            return pd.read_excel(path, **kwargs)

        elif suffix == ".csv":
            return pd.read_csv(path, **kwargs)

        elif suffix == ".tsv":
            return pd.read_csv(path, sep="\t", **kwargs)

        elif suffix == ".json":
            return pd.read_json(path, **kwargs)

        elif suffix == ".parquet":
            return pd.read_parquet(path, **kwargs)

        else:
            raise ValueError(f"Extension non supportee : {suffix}")


    def df_from_file_config(self, config : dict):
        reserved = {
            "type",
            "mode",
            "requires",
            "file",
            "sql",
            "target",
            "scenarios",
            "step_view",
            "order_by"
        }

        kwargs = {
            key: value
            for key, value in config.items()
            if key not in reserved
        }

        return self.df_from_file(config["file"], **kwargs)


    def should_process(self, name : str) -> bool:
        if name not in self.pipeline:
            raise KeyError(f"Objet absent du pipeline : {name}")

        config = self.pipeline[name]
        object_type = config["type"]
        mode = config.get("mode", "create_if_not_exists")

        if object_type in {"execute", "iteration"}:
            return True

        if object_type == "dataframe":
            return not self.table_view_exists(name)

        if object_type in {"table", "view"}:
            if mode == "create_or_replace":
                return True

            if mode == "create_if_not_exists":
                return not self.table_view_exists(name)

            raise ValueError(f"Mode non supporte pour {name} : {mode}")

        raise ValueError(f"Type non supporte pour {name} : {object_type}")


    def process_dataframe_type(self, name : str):
        config = self.pipeline[name]
        df = self.df_from_file_config(config)
        self.con.register(name, df)


    def process_table_view_type(self, name : str):
        config = self.pipeline[name]

        self.create_table_view(
            name=name,
            sql=config["sql"],
            type=config["type"],
            mode=config.get("mode", "create_if_not_exists")
        )


    def process_execute_type(self, name : str):
        config = self.pipeline[name]
        self.execute_sql(config["sql"])


    def replace_iteration_step_view(
        self,
        step_view : str,
        row : dict
    ):
        df_name = f"df_{step_view}"
        table_name = f"tmp_{step_view}"

        try:
            self.con.unregister(df_name)
        except Exception:
            pass

        step_df = pd.DataFrame([row])
        self.con.register(df_name, step_df)

        self.con.sql(
            f"""
            CREATE OR REPLACE TEMP TABLE {table_name} AS
            SELECT *
            FROM {df_name}
            """
        )

        self.con.sql(
            f"""
            CREATE OR REPLACE VIEW {step_view} AS
            SELECT *
            FROM {table_name}
            """
        )


    def process_iteration_type(self, name : str):
        config = self.pipeline[name]

        scenarios_name = config["scenarios"]
        step_view = config["step_view"]
        target = config["target"]
        order_by = config.get("order_by", [])

        order_sql = ""

        if order_by:
            order_sql = " ORDER BY " + ", ".join(order_by)

        rows = self.con.sql(
            f"""
            SELECT *
            FROM {scenarios_name}
            {order_sql}
            """
        ).df().to_dict(orient="records")

        for row in rows:
            self.replace_iteration_step_view(
                step_view=step_view,
                row=row
            )

            self.process_with_requires(
                target,
                processed=set()
            )


    def process(self, name : str):
        logging.getLogger().debug(f"process({name})")

        if name not in self.pipeline:
            raise KeyError(f"Objet absent du pipeline : {name}")

        config = self.pipeline[name]
        object_type = config["type"]

        if object_type == "dataframe":
            self.process_dataframe_type(name)

        elif object_type in {"table", "view"}:
            self.process_table_view_type(name)

        elif object_type == "execute":
            self.process_execute_type(name)

        elif object_type == "iteration":
            self.process_iteration_type(name)

        else:
            raise ValueError(f"Type non supporte : {object_type}")


    def process_with_requires(
        self,
        name : str,
        processed : set[str] | None = None
    ):
        if processed is None:
            processed = set()

        if name in processed:
            return

        if not self.should_process(name):
            processed.add(name)
            return

        config = self.pipeline[name]

        for subname in config.get("requires", []):
            self.process_with_requires(
                subname,
                processed
            )

        self.process(name)
        processed.add(name)


    def p_table_view(self, name : str):
        if name not in self.pipeline:
            raise KeyError(f"Objet absent du pipeline : {name}")

        if self.pipeline[name]["type"] in {"execute", "iteration"}:
            raise TypeError(
                f"{name} n'est pas une table ou une vue"
            )

        self.process_with_requires(name)
        return self.table_view(name)


    def p_execute(self, name : str):
        if name not in self.pipeline:
            raise KeyError(f"Objet absent du pipeline : {name}")

        if self.pipeline[name]["type"] != "execute":
            raise TypeError(f"{name} n'est pas une etape execute")

        self.process_with_requires(name)


    def p_iteration(self, name : str):
        if name not in self.pipeline:
            raise KeyError(f"Objet absent du pipeline : {name}")

        if self.pipeline[name]["type"] != "iteration":
            raise TypeError(f"{name} n'est pas une etape iteration")

        self.process_with_requires(name)


In [ ]:

class SimUtils():
    @classmethod
    def optimal_removals_approx(cls,
        counts: List[int],
        target_proportions: List[float],
        max_prop_error: float = 0.02,
        min_removals: int = 0,          # ← nouveau paramètre
        tol: float = 1e-9
    ) -> Optional[Tuple[List[int], List[int], int, List[float]]]:
        """
        Version assouplie + contrainte de retraits minimum.

        Paramètres supplémentaires
        --------------------------
        min_removals : int
            Nombre minimum d'éléments à retirer au total.
            Utile pour éviter la solution triviale (0 retrait) 
            quand la répartition initiale est déjà proche de la cible.
        """
        n = len(counts)
        if len(target_proportions) != n:
            raise ValueError("counts et target_proportions doivent avoir la même longueur")
        if not math.isclose(sum(target_proportions), 1.0, abs_tol=tol):
            raise ValueError("La somme des proportions cibles doit être égale à 1")
        if min_removals < 0:
            raise ValueError("min_removals doit être ≥ 0")

        total_initial = sum(counts)

        # Borne supérieure du total restant
        candidates = [
            math.floor(c / p + tol)
            for c, p in zip(counts, target_proportions)
            if p > tol
        ]
        max_t = min(candidates) if candidates else 0

        # On force un nombre minimum de retraits
        max_t = min(max_t, total_initial - min_removals)

        if max_t <= 0:
            return None

        for t in range(max_t, 0, -1):
            ideal = [p * t for p in target_proportions]

            # Arrondi initial plafonné
            remaining = [min(counts[i], max(0, int(round(ideal[i])))) for i in range(n)]
            current_sum = sum(remaining)

            # Ajustement pour atteindre exactement t
            while current_sum > t:
                candidates = [
                    (remaining[i] - ideal[i], i)
                    for i in range(n) if remaining[i] > 0
                ]
                if not candidates:
                    break
                _, idx = max(candidates)
                remaining[idx] -= 1
                current_sum -= 1

            while current_sum < t:
                candidates = [
                    (ideal[i] - remaining[i], i)
                    for i in range(n) if remaining[i] < counts[i]
                ]
                if not candidates:
                    break
                _, idx = max(candidates)
                remaining[idx] += 1
                current_sum += 1

            if current_sum != t:
                continue

            # Vérification de l'erreur
            actual_props = [r / t for r in remaining]
            max_err = max(abs(actual_props[i] - target_proportions[i]) for i in range(n))

            if max_err <= max_prop_error:
                removals = [counts[i] - remaining[i] for i in range(n)]
                return removals, remaining, t, actual_props

        return None

In [ ]:
cp = ConnectionPipeline("duckdb/pilotes/sim_v2/sim_v2.duckdb", "config")

In [ ]:
# ============================================================================
# REFERENTIELS
# ============================================================================
 
display(cp.p_table_view("t_hotel_codes"))
display(cp.p_table_view("t_machines"))
display(cp.p_table_view("t_types"))
display(cp.p_table_view("t_gammes"))
display(cp.p_table_view("t_categories"))
display(cp.p_table_view("t_natures"))
display(cp.p_table_view("t_marques"))
display(cp.p_table_view("t_fournisseurs"))

# ============================================================================
# RANKING GLOBAL DES NATURES
# ============================================================================

display(cp.p_table_view("t_rank_nature"))

# ============================================================================
# RANKING DES NATURES PAR GROUPE
# ============================================================================

display(cp.p_table_view("t_rank_nature_by_type"))
display(cp.p_table_view("t_rank_nature_by_gamme"))
display(cp.p_table_view("t_rank_nature_by_categorie"))
display(cp.p_table_view("t_rank_nature_by_marque"))
display(cp.p_table_view("t_rank_nature_by_fournisseur"))

# ============================================================================
# RANKING DES GROUPES
# ============================================================================

display(cp.p_table_view("t_rank_type"))
display(cp.p_table_view("t_rank_gamme"))
display(cp.p_table_view("t_rank_categorie"))
display(cp.p_table_view("t_rank_marque"))
display(cp.p_table_view("t_rank_fournisseur"))


# =============================================================================
# DATASETS DE RÉFÉRENCE ET D'OBSERVATION
# =============================================================================

display(cp.p_table_view("t_dataset_ref"))

display(cp.p_table_view("t_dataset_observation_total"))

display(cp.p_table_view("t_dataset_observation_par_mois"))


# =============================================================================
# MIX D'EXPOSITION EN NOMBRES
# =============================================================================

display(cp.p_table_view("t_dataset_mix_nombre_global"))

display(cp.p_table_view("t_dataset_mix_nombre_long"))

display(cp.p_table_view("t_dataset_mix_nombre_detail"))

display(cp.p_table_view("t_dataset_mix_nombre"))


# =============================================================================
# MIX D'EXPOSITION EN POURCENTAGES
# =============================================================================

display(cp.p_table_view("t_dataset_mix_pourcentage_global"))

display(cp.p_table_view("t_dataset_mix_pourcentage_long"))

display(cp.p_table_view("t_dataset_mix_pourcentage_detail"))

display(cp.p_table_view("t_dataset_mix_pourcentage"))


# =============================================================================
# DATASET FINAL À UNE LIGNE PAR HÔTEL
# =============================================================================

display(cp.p_table_view("t_dataset_pivot"))

In [ ]:
cp.table_view("df_sales")

In [ ]:
import json
import math
from pathlib import Path
from typing import Any, Iterable, Optional

import pandas as pd
import yaml


class ScenarioGenerator:
    GROUP_CONFIG = {
        "categorie": {
            "nature_table": "t_rank_nature_by_categorie",
            "group_table": "t_rank_categorie",
            "group_column": "categorie",
            "nature_rank_column": "rang_nature",
            "group_rank_column": "rang_categorie",
        },
        "gamme": {
            "nature_table": "t_rank_nature_by_gamme",
            "group_table": "t_rank_gamme",
            "group_column": "gamme",
            "nature_rank_column": "rang_nature",
            "group_rank_column": "rang_gamme",
        },
        "type": {
            "nature_table": "t_rank_nature_by_type",
            "group_table": "t_rank_type",
            "group_column": "type",
            "nature_rank_column": "rang_nature",
            "group_rank_column": "rang_type",
        },
        "marque": {
            "nature_table": "t_rank_nature_by_marque",
            "group_table": "t_rank_marque",
            "group_column": "marque",
            "nature_rank_column": "rang_nature",
            "group_rank_column": "rang_marque",
        },
        "fournisseur": {
            "nature_table": "t_rank_nature_by_fournisseur",
            "group_table": "t_rank_fournisseur",
            "group_column": "fournisseur",
            "nature_rank_column": "rang_nature",
            "group_rank_column": "rang_fournisseur",
        },
    }

    def __init__(
        self,
        cp,
        output_excel_path: str | Path = "data/scenarios.xlsx",
        output_pipeline_path: str | Path = "pipelines/4_scenarios_pipeline.yaml",
    ):
        self.cp = cp
        self.output_excel_path = Path(output_excel_path).expanduser().resolve()
        self.output_pipeline_path = Path(output_pipeline_path).expanduser().resolve()

        self.output_excel_path.parent.mkdir(parents=True, exist_ok=True)
        self.output_pipeline_path.parent.mkdir(parents=True, exist_ok=True)

        self._scenarios: dict[tuple[str, ...], None] = {}
        self.add_scenario([])

    @staticmethod
    def canonical_natures(natures: Iterable[Any]) -> tuple[str, ...]:
        values = {
            str(value).strip()
            for value in natures
            if value is not None
            and not pd.isna(value)
            and str(value).strip()
        }
        return tuple(sorted(values, key=lambda value: (value.casefold(), value)))

    @classmethod
    def list_value(cls, value: Any) -> list[str]:
        if value is None:
            return []

        if isinstance(value, (list, tuple, set)):
            return list(cls.canonical_natures(value))

        if hasattr(value, "tolist") and not isinstance(value, str):
            return list(cls.canonical_natures(value.tolist()))

        if isinstance(value, str):
            text = value.strip()

            if not text:
                return []

            try:
                parsed = json.loads(text)
                if isinstance(parsed, list):
                    return list(cls.canonical_natures(parsed))
            except json.JSONDecodeError:
                pass

            return [text]

        return [str(value)]

    def relation_df(self, name: str) -> pd.DataFrame:
        return self.cp.p_table_view(name).df()

    def add_scenario(self, natures: Iterable[Any]) -> bool:
        key = self.canonical_natures(natures)

        if key in self._scenarios:
            return False

        self._scenarios[key] = None
        return True

    def add_cumulative_values(
        self,
        values: Iterable[Any],
        include_full_removal: bool = True,
    ) -> int:
        ordered_values = [
            str(value).strip()
            for value in values
            if value is not None
            and not pd.isna(value)
            and str(value).strip()
        ]

        if not ordered_values:
            return 0

        stop = len(ordered_values)

        if not include_full_removal:
            stop = max(0, stop - 1)

        added = 0

        for size in range(1, stop + 1):
            if self.add_scenario(ordered_values[:size]):
                added += 1

        return added

    def add_global_nature_scenarios(
        self,
        table_name: str = "t_rank_nature",
        include_full_removal: bool = True,
    ) -> int:
        df = self.relation_df(table_name)

        required = {"hotel_code", "nature", "rang_nature"}
        missing = required.difference(df.columns)

        if missing:
            raise ValueError(
                f"Colonnes manquantes dans {table_name} : {sorted(missing)}"
            )

        added = 0

        for _, hotel_df in df.groupby("hotel_code", sort=True):
            hotel_df = hotel_df.sort_values(
                ["rang_nature", "nature"],
                kind="stable",
            )

            added += self.add_cumulative_values(
                hotel_df["nature"].tolist(),
                include_full_removal=include_full_removal,
            )

        return added

    def add_within_group_scenarios(
        self,
        group_name: str,
        include_full_removal: bool = True,
    ) -> int:
        config = self.GROUP_CONFIG[group_name]
        table_name = config["nature_table"]
        group_column = config["group_column"]
        rank_column = config["nature_rank_column"]

        df = self.relation_df(table_name)

        required = {
            "hotel_code",
            group_column,
            "nature",
            rank_column,
        }
        missing = required.difference(df.columns)

        if missing:
            raise ValueError(
                f"Colonnes manquantes dans {table_name} : {sorted(missing)}"
            )

        added = 0

        for _, group_df in df.groupby(
            ["hotel_code", group_column],
            sort=True,
            dropna=False,
        ):
            group_df = group_df.sort_values(
                [rank_column, "nature"],
                kind="stable",
            )

            added += self.add_cumulative_values(
                group_df["nature"].tolist(),
                include_full_removal=include_full_removal,
            )

        return added

    def add_ranked_group_scenarios(
        self,
        group_name: str,
        include_full_removal: bool = True,
    ) -> int:
        config = self.GROUP_CONFIG[group_name]
        table_name = config["group_table"]
        group_column = config["group_column"]
        rank_column = config["group_rank_column"]

        df = self.relation_df(table_name)

        required = {
            "hotel_code",
            group_column,
            "natures",
            rank_column,
        }
        missing = required.difference(df.columns)

        if missing:
            raise ValueError(
                f"Colonnes manquantes dans {table_name} : {sorted(missing)}"
            )

        added = 0

        for _, hotel_df in df.groupby("hotel_code", sort=True):
            hotel_df = hotel_df.sort_values(
                [rank_column, group_column],
                kind="stable",
            )

            cumulative: list[str] = []
            rows = list(hotel_df.itertuples(index=False))
            stop = len(rows)

            if not include_full_removal:
                stop = max(0, stop - 1)

            for row in rows[:stop]:
                cumulative.extend(
                    self.list_value(getattr(row, "natures"))
                )

                if self.add_scenario(cumulative):
                    added += 1

        return added

    def add_all_rank_scenarios(
        self,
        include_full_removal: bool = True,
        groups: Iterable[str] = (
            "categorie",
            "gamme",
            "type",
            "marque",
            "fournisseur",
        ),
    ) -> dict[str, int]:
        stats = {
            "global_nature": self.add_global_nature_scenarios(
                include_full_removal=include_full_removal,
            )
        }

        for group_name in groups:
            stats[f"within_{group_name}"] = self.add_within_group_scenarios(
                group_name,
                include_full_removal=include_full_removal,
            )

            stats[f"ranked_{group_name}"] = self.add_ranked_group_scenarios(
                group_name,
                include_full_removal=include_full_removal,
            )

        return stats

    def select_group_removals(
        self,
        group_name: str,
        hotel_code: str,
        removals_by_group: dict[str, int],
    ) -> list[str]:
        config = self.GROUP_CONFIG[group_name]
        table_name = config["nature_table"]
        group_column = config["group_column"]
        rank_column = config["nature_rank_column"]

        df = self.relation_df(table_name)
        hotel_df = df[df["hotel_code"] == hotel_code].copy()

        selected: list[str] = []

        for group_value, removal_count in removals_by_group.items():
            if removal_count <= 0:
                continue

            current = hotel_df[
                hotel_df[group_column].astype(str) == str(group_value)
            ].sort_values(
                [rank_column, "nature"],
                kind="stable",
            )

            selected.extend(
                current["nature"]
                .head(int(removal_count))
                .tolist()
            )

        return list(self.canonical_natures(selected))

    def add_mix_scenarios(
        self,
        group_name: str,
        target_proportions: dict[str, float],
        max_prop_error: float = 0.02,
        min_removals: int = 0,
    ) -> int:
        config = self.GROUP_CONFIG[group_name]
        table_name = config["nature_table"]
        group_column = config["group_column"]

        df = self.relation_df(table_name)
        target_groups = list(target_proportions.keys())
        proportions = [
            float(target_proportions[group])
            for group in target_groups
        ]

        added = 0

        for hotel_code, hotel_df in df.groupby("hotel_code", sort=True):
            counts_series = (
                hotel_df.groupby(group_column)["nature"]
                .nunique()
            )

            counts = [
                int(counts_series.get(group, 0))
                for group in target_groups
            ]

            result = SimUtils.optimal_removals_approx(
                counts=counts,
                target_proportions=proportions,
                max_prop_error=max_prop_error,
                min_removals=min_removals,
            )

            if result is None:
                continue

            removals, _, _, _ = result
            removals_by_group = dict(
                zip(target_groups, removals)
            )

            natures = self.select_group_removals(
                group_name=group_name,
                hotel_code=hotel_code,
                removals_by_group=removals_by_group,
            )

            if self.add_scenario(natures):
                added += 1

        return added

    def add_linear_meter_scenarios(
        self,
        group_name: str,
        target_meters: Optional[Iterable[float]] = None,
        max_prop_error: float = 0.02,
    ) -> int:
        config = self.GROUP_CONFIG[group_name]
        nature_table = config["nature_table"]
        group_column = config["group_column"]

        nature_df = self.relation_df(nature_table)
        hotel_df = self.cp.p_table_view("t_sales").df()

        hotel_ref = (
            hotel_df.groupby("HOTEL_CODE", as_index=False)
            .agg(
                metres_lineaires=("METRES_LINEAIRES", "max"),
            )
        )

        added = 0

        for row in hotel_ref.itertuples(index=False):
            hotel_code = row.HOTEL_CODE
            current_meters = float(row.metres_lineaires)

            if not math.isfinite(current_meters) or current_meters <= 0:
                continue

            current = nature_df[
                nature_df["hotel_code"] == hotel_code
            ].copy()

            counts_series = (
                current.groupby(group_column)["nature"]
                .nunique()
                .sort_index()
            )

            group_values = counts_series.index.tolist()
            counts = counts_series.astype(int).tolist()
            total_natures = sum(counts)

            if total_natures <= 1:
                continue

            proportions = [
                count / total_natures
                for count in counts
            ]

            if target_meters is None:
                last_meter = max(1, int(math.floor(current_meters)))
                meters = range(last_meter - 1, 0, -1)
            else:
                meters = sorted(
                    {
                        float(value)
                        for value in target_meters
                        if 0 < float(value) < current_meters
                    },
                    reverse=True,
                )

            for target_meter in meters:
                target_total = max(
                    1,
                    int(round(
                        total_natures
                        * target_meter
                        / current_meters
                    )),
                )

                min_removals = total_natures - target_total

                result = SimUtils.optimal_removals_approx(
                    counts=counts,
                    target_proportions=proportions,
                    max_prop_error=max_prop_error,
                    min_removals=min_removals,
                )

                if result is None:
                    continue

                removals, _, _, _ = result
                removals_by_group = dict(
                    zip(group_values, removals)
                )

                natures = self.select_group_removals(
                    group_name=group_name,
                    hotel_code=hotel_code,
                    removals_by_group=removals_by_group,
                )

                if self.add_scenario(natures):
                    added += 1

        return added

    def scenarios_df(self) -> pd.DataFrame:
        ordered = sorted(
            self._scenarios.keys(),
            key=lambda values: (
                len(values),
                tuple(value.casefold() for value in values),
                values,
            ),
        )

        if () in ordered:
            ordered.remove(())
        ordered.insert(0, ())

        return pd.DataFrame({
            "scenario_id": range(len(ordered)),
            "scenario_removed_natures_json": [
                json.dumps(
                    list(values),
                    ensure_ascii=False,
                )
                for values in ordered
            ],
        })

    def write_excel(self) -> Path:
        df = self.scenarios_df()
        df.to_excel(
            self.output_excel_path,
            index=False,
        )
        return self.output_excel_path

    def write_pipeline(self) -> Path:
        excel_path = self.output_excel_path.as_posix()

        pipeline = {
            "df_scenarios": {
                "type": "dataframe",
                "requires": [],
                "file": excel_path,
            },
            "t_scenarios": {
                "type": "table",
                "mode": "create_if_not_exists",
                "requires": ["df_scenarios"],
                "sql": """
SELECT
    CAST(scenario_id AS INTEGER) AS scenario_id,
    FROM_JSON(
        scenario_removed_natures_json,
        '["VARCHAR"]'
    ) AS scenario_removed_natures
FROM
    df_scenarios
ORDER BY
    scenario_id
""".strip(),
            },
        }

        self.output_pipeline_path.write_text(
            yaml.safe_dump(
                pipeline,
                sort_keys=False,
                allow_unicode=True,
                width=120,
            ),
            encoding="utf-8",
        )

        return self.output_pipeline_path

    def generate(
        self,
        include_full_removal: bool = True,
        add_rank_scenarios: bool = True,
    ) -> dict[str, Any]:
        stats: dict[str, Any] = {}

        if add_rank_scenarios:
            stats.update(
                self.add_all_rank_scenarios(
                    include_full_removal=include_full_removal,
                )
            )

        excel_path = self.write_excel()
        pipeline_path = self.write_pipeline()
        scenarios_df = self.scenarios_df()

        stats["scenario_count"] = len(scenarios_df)
        stats["excel_path"] = str(excel_path)
        stats["pipeline_path"] = str(pipeline_path)

        return stats


In [ ]:
scenario_generator = ScenarioGenerator(
    cp=cp,
    output_excel_path="data/scenarios.xlsx",
    output_pipeline_path="pipelines/4_scenarios_pipeline.yaml",
)

stats = scenario_generator.generate(
    include_full_removal=True,
)

display(scenario_generator.scenarios_df())
print(stats)
